# CS781 Phase 1 — Mamba / SSM

Standalone notebook using your verified 32K BPE + 4096 streaming corpus.

In [ ]:
from pathlib import Path
import os,json,random,math,zipfile
import numpy as np
import torch
from tokenizers import Tokenizer
from torch.utils.data import IterableDataset,DataLoader

INPUT=Path('/kaggle/input')
def find(name):
    hits=list(INPUT.rglob(name));
    if not hits: raise FileNotFoundError(name)
    return hits[0]
TOKENIZER_PATH=find('legal_bpe_32k.json')
TRAIN_MANIFEST=find('train_manifest.jsonl')
VAL_MANIFEST=find('validation_manifest.jsonl')
TEST_MANIFEST=find('test_manifest.jsonl')
INDEX_PATH=find('source_index.jsonl')
dedups=list(INPUT.rglob('dedup')); assert dedups, 'Attach cs781-phase4-training-prep'
DEDUP_ROOT=dedups[0]
RUN_DIR=Path('/kaggle/working/cs781_phase1_run'); RUN_DIR.mkdir(exist_ok=True)
CONTEXT_LENGTH=4096; VOCAB_SIZE=32000; SEED=42
BATCH_SIZE=1; GRAD_ACCUM_STEPS=8; NUM_WORKERS=0
LR=None; WEIGHT_DECAY=None; MAX_STEPS=None; MAX_GRAD_NORM=None
VAL_EVERY=250; SAVE_EVERY=250; MAX_VAL_BATCHES=20
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE=torch.bfloat16 if DEVICE.type=='cuda' and torch.cuda.is_bf16_supported() else torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('DEVICE',DEVICE); print('TOKENIZER',TOKENIZER_PATH); print('DEDUP',DEDUP_ROOT)


In [ ]:
tokenizer=Tokenizer.from_file(str(TOKENIZER_PATH))
assert tokenizer.get_vocab_size()==VOCAB_SIZE
PAD_ID=tokenizer.token_to_id('[PAD]'); BOS_ID=tokenizer.token_to_id('[BOS]'); EOS_ID=tokenizer.token_to_id('[EOS]')
assert None not in (PAD_ID,BOS_ID,EOS_ID)
source_lookup={}
with open(INDEX_PATH,encoding='utf-8') as f:
    for line in f:
        if line.strip():
            r=json.loads(line); source_lookup[r['text_hash']]=r

def read_record(rec):
    r=source_lookup[rec['text_hash']]; rel=str(r['source_file']); rel=rel[:-3] if rel.endswith('.gz') else rel
    p=DEDUP_ROOT/rel
    with open(p,'rb') as f: f.seek(int(r['byte_offset'])); raw=f.read(int(r['byte_length']))
    return json.loads(raw.rstrip(b'\r\n').decode())

def text_of(x):
    if isinstance(x,str): return x
    for k in ('text','content','document','body'):
        if isinstance(x.get(k),str): return x[k]
    raise KeyError(list(x))

class LegalStream(IterableDataset):
    def __init__(self,manifest):
        self.manifest=[json.loads(x) for x in open(manifest,encoding='utf-8') if x.strip()]
    def __iter__(self):
        for rec in self.manifest:
            ids=[BOS_ID]+tokenizer.encode(text_of(read_record(rec))).ids+[EOS_ID]
            for s in range(0,len(ids),CONTEXT_LENGTH):
                x=ids[s:s+CONTEXT_LENGTH]
                x=x+[PAD_ID]*(CONTEXT_LENGTH-len(x)) if len(x)<CONTEXT_LENGTH else x
                x=torch.tensor(x,dtype=torch.long); y=x.clone(); y[y==PAD_ID]=-100
                yield {'input_ids':x,'labels':y,'attention_mask':(x!=PAD_ID).long()}
train_ds=LegalStream(TRAIN_MANIFEST); val_ds=LegalStream(VAL_MANIFEST)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
print('vocab',tokenizer.get_vocab_size(),'train records',len(train_ds.manifest),'val records',len(val_ds.manifest))
print({k:tuple(v.shape) for k,v in next(iter(train_loader)).items()})


In [ ]:
# Mamba / SSM architecture
# Implement the selective state-space architecture from your course material.
# Implement this cell from your course lectures/pseudocode.
# Required API: model(input_ids=..., attention_mask=..., labels=...) -> object with .loss and .logits
# For generation, labels may be omitted. Initialize from scratch; do not load pretrained weights.

class ARCHITECTURE_TODO(torch.nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError('Implement Mamba / SSM here before training.')

# model=ARCHITECTURE_TODO()


## Training settings
These must come from your documented course-approved design; they are intentionally not invented here.

In [ ]:
LR=None
WEIGHT_DECAY=None
MAX_STEPS=None
MAX_GRAD_NORM=None

In [ ]:
LATEST=RUN_DIR/'latest.pt'; BEST=RUN_DIR/'best.pt'; FINAL=RUN_DIR/'final.pt'
def rng_state():
    return {'py':random.getstate(),'np':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
def restore(s):
    random.setstate(s['py']); np.random.set_state(s['np']); torch.set_rng_state(s['torch'])
    if torch.cuda.is_available() and s['cuda'] is not None: torch.cuda.set_rng_state_all(s['cuda'])
def save(path,step,seen,best):
    tmp=str(path)+'.tmp'; torch.save({'architecture':ARCH_NAME,'step':step,'seen':seen,'best':best,'model':model.state_dict(),'optimizer':opt.state_dict(),'scaler':scaler.state_dict(),'rng':rng_state()},tmp); os.replace(tmp,path)
def load(path):
    c=torch.load(path,map_location='cpu',weights_only=False)
    if c['architecture']!=ARCH_NAME: raise RuntimeError('wrong architecture checkpoint')
    model.load_state_dict(c['model']); opt.load_state_dict(c['optimizer']); scaler.load_state_dict(c['scaler']); restore(c['rng']); return c['step'],c['seen'],c['best']

if LR is None or WEIGHT_DECAY is None or MAX_STEPS is None or MAX_GRAD_NORM is None:
    raise RuntimeError('Set LR, WEIGHT_DECAY, MAX_STEPS and MAX_GRAD_NORM from your documented course-approved design.')
model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda' and AMP_DTYPE==torch.float16)
step=seen=0; best=float('inf')
if LATEST.exists(): step,seen,best=load(LATEST); print('RESUME',step,seen)
it=iter(train_loader)
for _ in range(seen): next(it)

def evaluate():
    model.eval(); vals=[]; vi=iter(val_loader)
    with torch.no_grad():
        for _ in range(MAX_VAL_BATCHES):
            try:b=next(vi)
            except StopIteration:break
            b={k:v.to(DEVICE) for k,v in b.items()}
            with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=DEVICE.type=='cuda'): o=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'],labels=b['labels'])
            vals.append(float(o.loss.cpu()))
    model.train(); return sum(vals)/len(vals) if vals else float('inf')

model.train(); opt.zero_grad(set_to_none=True)
while step<MAX_STEPS:
    for _ in range(GRAD_ACCUM_STEPS):
        try:b=next(it)
        except StopIteration:it=iter(train_loader); b=next(it)
        b={k:v.to(DEVICE) for k,v in b.items()}
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=DEVICE.type=='cuda'): loss=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'],labels=b['labels']).loss/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward(); seen+=1
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step+=1
    if step%10==0: print('step',step,'loss',float(loss.detach().cpu()))
    if step%VAL_EVERY==0:
        vl=evaluate(); print('val',vl)
        if vl<best: best=vl; save(BEST,step,seen,best)
    if step%SAVE_EVERY==0: save(LATEST,step,seen,best)
save(FINAL,step,seen,best); print('DONE',FINAL)


In [ ]:
# Official test generation / submission
# Keep the architecture's logits interface identical to training.
model.eval()
def generate(prompt_ids,max_new=384):
    ids=torch.tensor([prompt_ids],dtype=torch.long,device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new):
            x=ids[:,-CONTEXT_LENGTH:]; mask=(x!=PAD_ID).long()
            with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=DEVICE.type=='cuda'): out=model(input_ids=x,attention_mask=mask)
            nxt=torch.argmax(out.logits[:,-1,:],dim=-1,keepdim=True); ids=torch.cat([ids,nxt],1)
            if int(nxt)==EOS_ID: break
    return ids[0].tolist()
preds={}
with open(TEST_MANIFEST,encoding='utf-8') as f:
    for line in f:
        if not line.strip():continue
        rec=json.loads(line); prompt=text_of(read_record(rec)); p=tokenizer.encode(prompt).ids; new=generate([BOS_ID]+p); text=tokenizer.decode(new[len(p)+1:],skip_special_tokens=True); key=str(rec.get('id',rec.get('document_id',rec.get('text_hash')))); preds[key]={'text':' '.join(text.split()[:128])}
PRED=RUN_DIR/'prediction.json'; PRED.write_text(json.dumps(preds,ensure_ascii=False,indent=2),encoding='utf-8')
ZIP=RUN_DIR/'submission.zip';
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:z.write(PRED,arcname='prediction.json')
print(PRED); print(ZIP)


Outputs: latest.pt, best.pt, final.pt, prediction.json, submission.zip. Resume is automatic when latest.pt exists.